In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from vit_model import VisionTransformer
%load_ext autoreload
%autoreload 2

## Loading Cifar10 Data
- Performing transformation to tensor
- Performing normalization for the rgb channels to zero


In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f'Number of training batches: {len(train_loader)}')

  1%|          | 1.28M/170M [00:51<1:54:23, 24.7kB/s]


KeyboardInterrupt: 

In [6]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=6)


100%|██████████| 170M/170M [52:11<00:00, 54.4kB/s]   


In [ ]:
import torch.optim as optim
from tqdm import tqdm

device = torch.device('cuda' if torch.cude.is_available() else 'cpu')
print(f'Training on : {device}')

model = VisionTransformer(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-4)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')

    for images, labels in progress_bar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        progress_bar.set_postfix({'loss': running_loss / (progress_bar.n + 1)})

print('Training Complete')
torch.save(model.state_dict(), 'vit_cifar10_weights.pth')

# Model Evaluation

In [3]:
import torch
import torch.nn as nn
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Evaluating on: {device}')

model = VisionTransformer(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()

weights_path = 'vit_cifar10_weights.pth'
model.load_state_dict(torch.load(weights_path, map_location=device))
print(f'Model weights loaded from {weights_path}')



Evaluating on: cpu
Model weights loaded from vit_cifar10_weights.pth


In [7]:
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    test_bar = tqdm(test_loader, desc='Evaluating Model')
    for images, labels in test_bar:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        test_bar.set_postfix({'loss':test_loss / (test_bar.n + 1)})
accuracy = 100 * correct / total
avg_test_loss = test_loss / len(test_loader)

print('\n---- Evaluation Results ----')
print(f'Test Loss: {avg_test_loss:.4f}')
print(f'Test Accuracy: {accuracy:.2f}%')

Evaluating Model: 100%|██████████| 79/79 [01:07<00:00,  1.16it/s, loss=1.05]


---- Evaluation Results ----
Test Loss: 1.0465
Test Accuracy: 63.98%


In [9]:
import torch
import torch.nn as nn
from tqdm import tqdm
from cnn_vit_hybrid_model import HybridVisionTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Evaluating on: {device}')

model = HybridVisionTransformer(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()

weights_path = 'hybrid_cifar10_weights.pth'
model.load_state_dict(torch.load(weights_path, map_location=device))
print(f'Model weights loaded from {weights_path}')



Evaluating on: cpu
Model weights loaded from hybrid_cifar10_weights.pth


In [10]:
model.eval()
test_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    test_bar = tqdm(test_loader, desc='Evaluating Model')
    for images, labels in test_bar:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        test_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        test_bar.set_postfix({'loss':test_loss / (test_bar.n + 1)})
accuracy = 100 * correct / total
avg_test_loss = test_loss / len(test_loader)

print('\n---- Evaluation Results ----')
print(f'Test Loss: {avg_test_loss:.4f}')
print(f'Test Accuracy: {accuracy:.2f}%')

Evaluating Model: 100%|██████████| 79/79 [00:54<00:00,  1.46it/s, loss=0.69] 


---- Evaluation Results ----
Test Loss: 0.6898
Test Accuracy: 78.71%
